In [2]:
import numpy as np
import pandas as pd

In [4]:
train_df = pd.read_csv("NBA_Train.csv")
val_df = pd.read_csv("NBA_Validation.csv")
test_df = pd.read_csv("NBA_Test.csv")
TARGET_COL = "draft_status"
DROP_COLS = ["player_name", "pid", "year"]   # do not use in training
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [5]:
def one_hot(y, num_classes):
    y = y.astype(int)
    out = np.zeros((len(y), num_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


# =========================================================
# 4. Prepare features
#    If categorical columns still exist, one-hot encode them
# =========================================================
def prepare_data(train_df, val_df, test_df, target_col, drop_cols):
    # Copy
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    # Separate y
    y_train = train_df[target_col].values.astype(int)
    y_val = val_df[target_col].values.astype(int)
    y_test = test_df[target_col].values.astype(int)

    # Drop target + forbidden cols
    cols_to_drop = [target_col] + [c for c in drop_cols if c in train_df.columns]

    X_train_df = train_df.drop(columns=cols_to_drop, errors="ignore")
    X_val_df = val_df.drop(columns=cols_to_drop, errors="ignore")
    X_test_df = test_df.drop(columns=cols_to_drop, errors="ignore")

    # Combine first so dummy columns align
    combined = pd.concat(
        [
            X_train_df.assign(__split__="train"),
            X_val_df.assign(__split__="val"),
            X_test_df.assign(__split__="test"),
        ],
        axis=0,
        ignore_index=True,
    )

    # One-hot encode any remaining categorical columns
    combined_encoded = pd.get_dummies(combined, drop_first=False)

    # Split back
    train_encoded = combined_encoded[combined_encoded["__split___train"] == 1].drop(
        columns=[c for c in combined_encoded.columns if c.startswith("__split__")],
        errors="ignore"
    )
    val_encoded = combined_encoded[combined_encoded["__split___val"] == 1].drop(
        columns=[c for c in combined_encoded.columns if c.startswith("__split__")],
        errors="ignore"
    )
    test_encoded = combined_encoded[combined_encoded["__split___test"] == 1].drop(
        columns=[c for c in combined_encoded.columns if c.startswith("__split__")],
        errors="ignore"
    )

    X_train = train_encoded.to_numpy(dtype=float)
    X_val = val_encoded.to_numpy(dtype=float)
    X_test = test_encoded.to_numpy(dtype=float)

    return X_train, y_train, X_val, y_val, X_test, y_test, train_encoded.columns.tolist()


# =========================================================
# 5. Class weights for imbalance
#    balanced: N / (K * n_c)
# =========================================================
def compute_class_weights(y, num_classes):
    counts = np.bincount(y, minlength=num_classes)
    n = len(y)
    weights = np.zeros(num_classes)

    for c in range(num_classes):
        if counts[c] == 0:
            weights[c] = 0.0
        else:
            weights[c] = n / (num_classes * counts[c])

    return weights


# =========================================================
# 6. Softmax model
# =========================================================
def softmax(logits):
    # numerical stability
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


def compute_loss_and_gradients(X, y, W, b, reg_lambda=0.0, class_weights=None):
    """
    X: (n, d)
    y: (n,)
    W: (d, k)
    b: (1, k)
    """
    n = X.shape[0]
    k = W.shape[1]

    logits = X @ W + b
    probs = softmax(logits)

    Y = one_hot(y, k)

    # Sample weights from class weights
    if class_weights is None:
        sample_weights = np.ones(n)
    else:
        sample_weights = class_weights[y]

    # Weighted cross-entropy
    eps = 1e-12
    correct_log_probs = -np.sum(Y * np.log(probs + eps), axis=1)
    data_loss = np.sum(sample_weights * correct_log_probs) / np.sum(sample_weights)

    reg_loss = reg_lambda * np.sum(W * W)
    loss = data_loss + reg_loss

    # Gradient
    diff = probs - Y  # (n, k)
    diff = diff * sample_weights[:, None]
    diff /= np.sum(sample_weights)

    dW = X.T @ diff + 2 * reg_lambda * W
    db = np.sum(diff, axis=0, keepdims=True)

    return loss, dW, db


def predict_proba(X, W, b):
    logits = X @ W + b
    return softmax(logits)


def predict(X, W, b):
    probs = predict_proba(X, W, b)
    return np.argmax(probs, axis=1)


# =========================================================
# 7. Metrics
# =========================================================
def confusion_matrix_manual(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def precision_recall_f1_per_class(y_true, y_pred, num_classes):
    cm = confusion_matrix_manual(y_true, y_pred, num_classes)

    precision_list = []
    recall_list = []
    f1_list = []

    for c in range(num_classes):
        tp = cm[c, c]
        fp = np.sum(cm[:, c]) - tp
        fn = np.sum(cm[c, :]) - tp

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)

        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)

    return np.array(precision_list), np.array(recall_list), np.array(f1_list)


def accuracy_score_manual(y_true, y_pred):
    return np.mean(y_true == y_pred)


def macro_metrics(y_true, y_pred, num_classes):
    precision_c, recall_c, f1_c = precision_recall_f1_per_class(y_true, y_pred, num_classes)

    return {
        "accuracy": accuracy_score_manual(y_true, y_pred),
        "macro_precision": np.mean(precision_c),
        "macro_recall": np.mean(recall_c),
        "macro_f1": np.mean(f1_c),
        "precision_per_class": precision_c,
        "recall_per_class": recall_c,
        "f1_per_class": f1_c,
        "confusion_matrix": confusion_matrix_manual(y_true, y_pred, num_classes),
    }


# =========================================================
# 8. Mini-batch training
# =========================================================
def train_softmax_regression(
    X_train,
    y_train,
    X_val,
    y_val,
    num_classes,
    lr=0.01,
    reg_lambda=1e-4,
    epochs=200,
    batch_size=128,
    use_class_weights=True,
    verbose=True
):
    n, d = X_train.shape

    W = np.zeros((d, num_classes))
    b = np.zeros((1, num_classes))

    class_weights = compute_class_weights(y_train, num_classes) if use_class_weights else None

    history = {
        "train_loss": [],
        "val_macro_f1": [],
        "val_macro_recall": [],
        "val_accuracy": []
    }

    best_W = W.copy()
    best_b = b.copy()
    best_val_f1 = -np.inf

    for epoch in range(epochs):
        # Shuffle each epoch
        indices = np.random.permutation(n)
        X_train_shuffled = X_train[indices]
        y_train_shuffled = y_train[indices]

        # Mini-batch GD
        for start in range(0, n, batch_size):
            end = start + batch_size
            X_batch = X_train_shuffled[start:end]
            y_batch = y_train_shuffled[start:end]

            loss, dW, db = compute_loss_and_gradients(
                X_batch, y_batch, W, b,
                reg_lambda=reg_lambda,
                class_weights=class_weights
            )

            W -= lr * dW
            b -= lr * db

        # End-of-epoch evaluation
        train_loss, _, _ = compute_loss_and_gradients(
            X_train, y_train, W, b,
            reg_lambda=reg_lambda,
            class_weights=class_weights
        )

        y_val_pred = predict(X_val, W, b)
        val_result = macro_metrics(y_val, y_val_pred, num_classes)

        history["train_loss"].append(train_loss)
        history["val_macro_f1"].append(val_result["macro_f1"])
        history["val_macro_recall"].append(val_result["macro_recall"])
        history["val_accuracy"].append(val_result["accuracy"])

        if val_result["macro_f1"] > best_val_f1:
            best_val_f1 = val_result["macro_f1"]
            best_W = W.copy()
            best_b = b.copy()

        if verbose and (epoch % 10 == 0 or epoch == epochs - 1):
            print(
                f"Epoch {epoch:03d} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Acc: {val_result['accuracy']:.4f} | "
                f"Val Macro-F1: {val_result['macro_f1']:.4f} | "
                f"Val Macro-Recall: {val_result['macro_recall']:.4f}"
            )

    return best_W, best_b, history


# =========================================================
# 9. Hyperparameter tuning
# =========================================================
def grid_search_softmax(X_train, y_train, X_val, y_val, num_classes):
    param_grid = {
        "lr": [0.001, 0.005, 0.01],
        "reg_lambda": [0.0, 1e-4, 1e-3, 1e-2],
        "batch_size": [64, 128, 256],
        "epochs": [100, 200],
        "use_class_weights": [True, False],
    }

    best_result = None
    best_score = -np.inf
    best_model = None

    for lr in param_grid["lr"]:
        for reg_lambda in param_grid["reg_lambda"]:
            for batch_size in param_grid["batch_size"]:
                for epochs in param_grid["epochs"]:
                    for use_class_weights in param_grid["use_class_weights"]:
                        print("\n================================================")
                        print(
                            f"Trying lr={lr}, reg_lambda={reg_lambda}, "
                            f"batch_size={batch_size}, epochs={epochs}, "
                            f"class_weights={use_class_weights}"
                        )

                        W, b, history = train_softmax_regression(
                            X_train, y_train, X_val, y_val,
                            num_classes=num_classes,
                            lr=lr,
                            reg_lambda=reg_lambda,
                            epochs=epochs,
                            batch_size=batch_size,
                            use_class_weights=use_class_weights,
                            verbose=False
                        )

                        y_val_pred = predict(X_val, W, b)
                        result = macro_metrics(y_val, y_val_pred, num_classes)

                        print(
                            f"Val Acc={result['accuracy']:.4f}, "
                            f"Val Macro-F1={result['macro_f1']:.4f}, "
                            f"Val Macro-Recall={result['macro_recall']:.4f}"
                        )

                        if result["macro_f1"] > best_score:
                            best_score = result["macro_f1"]
                            best_result = {
                                "lr": lr,
                                "reg_lambda": reg_lambda,
                                "batch_size": batch_size,
                                "epochs": epochs,
                                "use_class_weights": use_class_weights,
                                "val_metrics": result
                            }
                            best_model = (W, b)

    return best_result, best_model

In [6]:
X_train, y_train, X_val, y_val, X_test, y_test, feature_names = prepare_data(
    train_df, val_df, test_df, TARGET_COL, DROP_COLS
)

num_classes = len(np.unique(y_train))

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("Classes:", np.unique(y_train))
print("Feature count:", len(feature_names))

# Tune on validation
best_result, best_model = grid_search_softmax(X_train, y_train, X_val, y_val, num_classes)

print("\n================ BEST VALIDATION RESULT ================")
print(best_result)

best_W, best_b = best_model

# Final test evaluation
y_test_pred = predict(X_test, best_W, best_b)
test_result = macro_metrics(y_test, y_test_pred, num_classes)

print("\n================ TEST RESULT ================")
print(f"Test Accuracy      : {test_result['accuracy']:.4f}")
print(f"Test Macro Precision: {test_result['macro_precision']:.4f}")
print(f"Test Macro Recall   : {test_result['macro_recall']:.4f}")
print(f"Test Macro F1       : {test_result['macro_f1']:.4f}")
print("Precision per class :", test_result["precision_per_class"])
print("Recall per class    :", test_result["recall_per_class"])
print("F1 per class        :", test_result["f1_per_class"])
print("Confusion Matrix:\n", test_result["confusion_matrix"])

X_train shape: (16846, 467)
X_val shape: (3643, 467)
X_test shape: (4956, 467)
Classes: [0 1 2]
Feature count: 467

Trying lr=0.001, reg_lambda=0.0, batch_size=64, epochs=100, class_weights=True
Val Acc=0.9528, Val Macro-F1=0.5220, Val Macro-Recall=0.6096

Trying lr=0.001, reg_lambda=0.0, batch_size=64, epochs=100, class_weights=False
Val Acc=0.9728, Val Macro-F1=0.5485, Val Macro-Recall=0.5194

Trying lr=0.001, reg_lambda=0.0, batch_size=64, epochs=200, class_weights=True
Val Acc=0.9432, Val Macro-F1=0.5399, Val Macro-Recall=0.6845

Trying lr=0.001, reg_lambda=0.0, batch_size=64, epochs=200, class_weights=False
Val Acc=0.9572, Val Macro-F1=0.5445, Val Macro-Recall=0.6339

Trying lr=0.001, reg_lambda=0.0, batch_size=128, epochs=100, class_weights=True
Val Acc=0.9520, Val Macro-F1=0.5369, Val Macro-Recall=0.6305

Trying lr=0.001, reg_lambda=0.0, batch_size=128, epochs=100, class_weights=False
Val Acc=0.9498, Val Macro-F1=0.5289, Val Macro-Recall=0.6521

Trying lr=0.001, reg_lambda=0.0, 